In [12]:
import torch
from torch import nn

In [13]:
class SoftmaxRegression(nn.Module):
    
    def __init__(
        self,
        num_outputs,
        learning_rate,
    ):
        super().__init__()
        
        self.learning_rate = learning_rate
        self.net = nn.Sequential(
            
            # model.net[0]
            # - [B, C, H, W] -> [B, C * H * W]
            nn.Flatten(),
            
            # model.net[1]
            # - 첫 forward에서 입력 feature 수를 자동으로 결정한다
            nn.LazyLinear(num_outputs),
        )
        
    def forward(self, images):
        # 확률이 아닌 logit를 반환
        return self.net(images)

In [14]:
model = SoftmaxRegression(
    num_outputs=10,
    learning_rate=0.1
)

print(model)

SoftmaxRegression(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): LazyLinear(in_features=0, out_features=10, bias=True)
  )
)


In [15]:
# Fashion-MNIST 형식의 가상 이미지 4개
demo_images = torch.randn(
    4,
    1,
    28,
    28,
)

flattened_images = model.net[0](
    demo_images,
)

print("Images shape:", demo_images.shape)
print("Flattened images shape:", flattened_images.shape)

Images shape: torch.Size([4, 1, 28, 28])
Flattened images shape: torch.Size([4, 784])


In [16]:
logits = model(
    demo_images,
)

print("Logits:")
print(logits)

print("\nLogits shape:")
print(logits.shape)


Logits:
tensor([[ 0.8536,  0.4603,  0.3815,  0.6701,  0.0393, -0.2540,  1.4938,  0.1128,
          0.3414, -0.4889],
        [-0.3478, -0.7287,  0.4408, -0.7062,  0.3953,  0.4105, -0.6832,  0.7749,
          0.8357, -0.4469],
        [-0.1532, -0.5669,  0.4507, -0.3603, -0.0246,  0.0393, -0.5241, -0.2550,
         -0.1521, -0.3726],
        [ 0.8686,  0.2843,  0.0395,  0.7500,  0.3470, -0.4105, -0.3306, -1.1946,
         -0.4559, -0.0137]], grad_fn=<AddmmBackward0>)

Logits shape:
torch.Size([4, 10])


In [ ]:
linear_layer = model.net[1]

print("Input features:", linear_layer.in_features)
print("Output features:", linear_layer.out_features)

# PyTorch의 nn.Linear는 가중치를 아래의 shape으로 저장한다.
# -> [out_features, in_features]
print("\nWeight:", linear_layer.weight.shape)
print(linear_layer.weight)

print("\nBias:", linear_layer.bias.shape)
print(linear_layer.bias)

Input features: 784
Output features: 10

Weight: torch.Size([10, 784])
Parameter containing:
tensor([[-0.0014,  0.0095, -0.0303,  ...,  0.0099,  0.0045, -0.0306],
        [ 0.0219,  0.0026,  0.0132,  ...,  0.0177, -0.0050,  0.0038],
        [-0.0125,  0.0159,  0.0328,  ..., -0.0036,  0.0214,  0.0345],
        ...,
        [ 0.0171, -0.0267, -0.0157,  ...,  0.0263, -0.0154, -0.0064],
        [ 0.0015,  0.0094, -0.0311,  ...,  0.0078, -0.0061,  0.0066],
        [-0.0333,  0.0124,  0.0207,  ...,  0.0122, -0.0288, -0.0050]],
       requires_grad=True)

Bias: torch.Size([10])
Parameter containing:
tensor([ 0.0153,  0.0090, -0.0349,  0.0120,  0.0287,  0.0145,  0.0013, -0.0143,
        -0.0284,  0.0045], requires_grad=True)
torch.Size([4, 784])


In [ ]:
manual_logits = (
    flattened_images
    @ linear_layer.weight.T
    + linear_layer.bias
)

print("Model logits:")
print(logits)

print("\nManual logits:")
print(manual_logits)

assert torch.allclose(
    logits,
    manual_logits,
)

Model logits:
tensor([[ 0.8536,  0.4603,  0.3815,  0.6701,  0.0393, -0.2540,  1.4938,  0.1128,
          0.3414, -0.4889],
        [-0.3478, -0.7287,  0.4408, -0.7062,  0.3953,  0.4105, -0.6832,  0.7749,
          0.8357, -0.4469],
        [-0.1532, -0.5669,  0.4507, -0.3603, -0.0246,  0.0393, -0.5241, -0.2550,
         -0.1521, -0.3726],
        [ 0.8686,  0.2843,  0.0395,  0.7500,  0.3470, -0.4105, -0.3306, -1.1946,
         -0.4559, -0.0137]], grad_fn=<AddmmBackward0>)

Manual logits:
tensor([[ 0.8536,  0.4603,  0.3815,  0.6701,  0.0393, -0.2540,  1.4938,  0.1128,
          0.3414, -0.4889],
        [-0.3478, -0.7287,  0.4408, -0.7062,  0.3953,  0.4105, -0.6832,  0.7749,
          0.8357, -0.4469],
        [-0.1532, -0.5669,  0.4507, -0.3603, -0.0246,  0.0393, -0.5241, -0.2550,
         -0.1521, -0.3726],
        [ 0.8686,  0.2843,  0.0395,  0.7500,  0.3470, -0.4105, -0.3306, -1.1946,
         -0.4559, -0.0137]], grad_fn=<AddBackward0>)
